# Pangaea Benchmark — Results Analysis
Load results from local experiment directories and build pivot tables.

In [ ]:
import re
import os
from pathlib import Path
import pandas as pd
import numpy as np

EXPERIMENTS_BASE = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments")

# ── Display name mappings ────────────────────────────────────────────────────
ENCODER_LABELS = {
    "carl_no_internal_norm": "CARL",
    "dofa":               "DOFA",
    "dofav2":             "DOFAv2",
    "carl":               "CARL",
    "panopticon":         "Panopticon",
    "copernicus_fm_base": "Copernicus-FM",
    "smarties_vitb":      "SMARTIES",
    "dinov2_vitb14":      "DINOv2",
    "unet_encoder":       "UNet",
    "unet_encoder_mi":    "UNet",
    "vit_scratch":        "ViT-Scratch",
    "vit_scratch_mi":     "ViT-Scratch",
    "vit_scratch_p8":     "ViT-Scratch-p8",
    "vit_scratch_mi_p8":  "ViT-Scratch-p8",
}

DATASET_LABELS = {
    "hlsburnscars":      "HLS-BurnScars",
    "sen1floods11":      "Sen1Floods11",
    "mados":             "MADOS",
    "pastis":            "PASTIS",
    "dynamicen":         "DynamicEarthNet",
    "croptypemapping":   "CropTypeMapping",
    "spacenet7cd":       "SpaceNet7-CD",
    "ai4smallfarms":     "AI4SmallFarms",
    "fivebillionpixels": "FiveBillionPixels",
}

DATASET_ORDER = list(DATASET_LABELS.keys())

# Encoders listed in longest-first order for greedy name matching
KNOWN_ENCODERS = [
    "copernicus_fm_base", "copernicus_fm_large",
    "carl_no_internal_norm",
    "smarties_vitb", "dinov2_vitb14",
    "dofav2", "dofa", "panopticon", "carl",
    "vit_scratch_mi_p8", "vit_scratch_p8",
    "vit_scratch_mi", "vit_scratch",
    "unet_encoder_mi", "unet_encoder",
]

KNOWN_DECODERS = [
    "seg_upernet_mt_ltae", "seg_upernet",
    "seg_siamupernet_conc", "seg_siamupernet_diff",
    "seg_siam_mlp_fusion", "seg_mlp_fusion",
    "seg_siamunet_conc", "seg_siamunet_diff", "seg_unet",
    "seg_linear_probe", "seg_siam_linear_probe",
    "cls_linear_probe",
]

DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in DATASET_ORDER) + r")_"
)

print("Config loaded.")

In [32]:
# ── Core parsing functions ───────────────────────────────────────────────────

def parse_run_dir_name(name: str) -> dict:
    """Extract dataset, encoder, decoder from a run directory name."""
    m = DATASET_RE.search(name)
    if not m:
        return {}
    dataset = m.group(1)
    rest = name[m.end():]

    encoder = next((e for e in KNOWN_ENCODERS if rest.startswith(e + "_")), None)
    if encoder:
        rest = rest[len(encoder) + 1:]

    decoder = next((d for d in KNOWN_DECODERS if rest.startswith(d)), None)

    return {"dataset": dataset, "encoder": encoder or "", "decoder": decoder or ""}


def parse_log_metrics(log_path: Path) -> dict:
    """
    Parse train.log-0 for test metrics.

    Extracts the Mean values from [test] IoU, F1, Precision, Recall sections.
    Returns a dict with keys: test_mIoU, test_mF1, test_mPrecision, test_mRecall.
    For change detection (IoU section contains exactly 2 classes), also returns test_IoU.
    """
    if not log_path.exists():
        return {}

    text = log_path.read_text(errors="ignore")

    # Find all [test] sections — there may be multiple if val was run at test time
    # We want the LAST occurrence (final test run)
    sections = re.split(r"\[test\] ------- IoU", text)
    if len(sections) < 2:
        return {}
    test_block = "[test] ------- IoU" + sections[-1]

    metrics = {}
    metric_map = {
        "IoU":       "test_mIoU",
        "F1-score":  "test_mF1",
        "Precision": "test_mPrecision",
        "Recall":    "test_mRecall",
    }

    for section_name, metric_key in metric_map.items():
        # Find the Mean line inside this section
        pattern = rf"\[test\] ------- {section_name}.*?\[test\] Mean\s+([\d.]+)"
        m = re.search(pattern, test_block, re.DOTALL)
        if m:
            metrics[metric_key] = float(m.group(1))

    return metrics


def load_experiment_folder(
    folder: str | Path,
    encoder_map: dict | None = None,
    dataset_map: dict | None = None,
    metric: str = "test_mIoU",
    keep_all_metrics: bool = False,
) -> pd.DataFrame:
    """
    Load all runs from an experiment folder and return a long-form DataFrame.

    Parameters
    ----------
    folder : path to an experiment directory (e.g. pangaea_full_seg/upernet/panopticon_seg)
             OR a parent directory containing multiple encoder subdirectories.
             The function auto-detects whether run dirs are direct children or
             one level deeper.
    encoder_map : optional override for encoder display names {raw: display}
    dataset_map : optional override for dataset display names {raw: display}
    metric : which metric column to extract (default: test_mIoU)
    keep_all_metrics : if True, keep all parsed metrics instead of just `metric`

    Returns
    -------
    Long-form DataFrame with columns: run_dir, encoder, decoder, dataset, [metrics...]
    """
    enc_map = {**ENCODER_LABELS, **(encoder_map or {})}
    ds_map  = {**DATASET_LABELS,  **(dataset_map  or {})}
    folder  = Path(folder)

    # Collect candidate run directories:
    # A run dir contains train.log-0. Walk up to 2 levels deep.
    run_dirs = []
    for child in sorted(folder.iterdir()):
        if not child.is_dir():
            continue
        if (child / "train.log-0").exists():
            run_dirs.append(child)
        else:
            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and (grandchild / "train.log-0").exists():
                    run_dirs.append(grandchild)

    rows = []
    for run_dir in run_dirs:
        parsed = parse_run_dir_name(run_dir.name)
        if not parsed.get("dataset"):
            continue
        log_metrics = parse_log_metrics(run_dir / "train.log-0")
        if not log_metrics:
            continue

        enc_raw = parsed["encoder"]
        ds_raw  = parsed["dataset"]

        row = {
            "run_dir": run_dir.name,
            "encoder_raw": enc_raw,
            "decoder":     parsed["decoder"],
            "dataset_raw": ds_raw,
            "encoder":     enc_map.get(enc_raw, enc_raw),
            "dataset":     ds_map.get(ds_raw, ds_raw),
        }
        if keep_all_metrics:
            row.update(log_metrics)
        elif metric in log_metrics:
            row[metric] = log_metrics[metric]
        else:
            continue
        rows.append(row)

    return pd.DataFrame(rows)


def to_pivot(
    df: pd.DataFrame,
    metric: str = "test_mIoU",
    index: str = "encoder",
    columns: str = "dataset",
    aggfunc: str = "max",
    dataset_order: list | None = None,
    encoder_order: list | None = None,
    add_mean: bool = True,
) -> pd.DataFrame:
    """
    Pivot a long-form results DataFrame to encoder × dataset.

    Parameters
    ----------
    df           : output of load_experiment_folder()
    metric       : value column to pivot
    index        : row grouping (default: encoder display name)
    columns      : column grouping (default: dataset display name)
    aggfunc      : how to aggregate duplicates — 'max', 'mean', or 'last'
    dataset_order: explicit column order; defaults to DATASET_LABELS order
    encoder_order: explicit row order
    add_mean     : whether to add a Mean column (over available values)
    """
    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not in DataFrame columns: {list(df.columns)}")

    pivot = df.pivot_table(
        index=index, columns=columns, values=metric,
        aggfunc=aggfunc
    )

    # Apply column order
    if dataset_order is None:
        ordered_ds = [DATASET_LABELS[d] for d in DATASET_ORDER if DATASET_LABELS[d] in pivot.columns]
        remaining  = [c for c in pivot.columns if c not in ordered_ds]
        dataset_order = ordered_ds + remaining
    pivot = pivot.reindex(columns=[c for c in dataset_order if c in pivot.columns])

    # Apply row order
    if encoder_order is not None:
        pivot = pivot.reindex([e for e in encoder_order if e in pivot.index])

    if add_mean:
        pivot["Mean"] = pivot.mean(axis=1, skipna=True).round(3)

    return pivot.round(2)


print("Functions defined: load_experiment_folder(), to_pivot()")

Functions defined: load_experiment_folder(), to_pivot()


## Full-band UPerNet results

In [ ]:
_VIT_FROZEN_MAP = {
    "vit_scratch":       "ViT-Frozen",
    "vit_scratch_mi":    "ViT-Frozen",
    "vit_scratch_p8":    "ViT-Frozen-p8",
    "vit_scratch_mi_p8": "ViT-Frozen-p8",
}

upernet_path = EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet"
_dfs = []
for sf in sorted(upernet_path.iterdir()):
    if not sf.is_dir():
        continue
    enc_map = _VIT_FROZEN_MAP if "frozen" in sf.name else None
    _dfs.append(load_experiment_folder(sf, encoder_map=enc_map, metric="test_mIoU"))

df_upernet = pd.concat(_dfs, ignore_index=True)
print(f"{len(df_upernet)} runs loaded")
df_upernet.groupby(["encoder", "dataset"])["test_mIoU"].max().unstack()

In [34]:
pivot_upernet = to_pivot(df_upernet)
pivot_upernet

dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
CARL,80.84,88.17,50.50,37.78,37.38,44.66,58.90,33.74,50.32,53.59
Copernicus-FM,82.33,89.11,56.90,33.78,37.72,48.08,62.36,33.35,62.80,56.27
DINOv2,74.48,70.02,58.78,18.78,26.58,36.73,62.84,35.20,64.58,49.78
DOFA,80.64,87.69,60.39,30.37,41.12,36.21,61.07,36.44,69.93,55.98
DOFAv2,78.27,88.01,48.60,19.79,42.94,36.23,63.63,38.08,71.22,54.09
Panopticon,84.22,90.59,68.22,36.81,40.55,47.99,60.77,37.88,70.10,59.68
SMARTIES,82.84,90.05,62.28,35.75,39.41,52.04,59.51,35.47,57.32,57.18
UNet,85.49,91.04,53.67,26.84,39.47,36.71,62.02,46.34,65.59,56.35
ViT-Frozen,80.84,89.42,47.56,9.16,31.56,45.06,59.97,33.28,48.34,49.46


## Full-band MLP Fusion results

In [35]:
mlp_path = EXPERIMENTS_BASE / "pangaea_full_seg" / "mlp_fusion"
_dfs = []
for sf in sorted(mlp_path.iterdir()):
    if not sf.is_dir():
        continue
    enc_map = _VIT_FROZEN_MAP if "frozen" in sf.name else None
    _dfs.append(load_experiment_folder(sf, encoder_map=enc_map, metric="test_mIoU"))

df_mlp = pd.concat(_dfs, ignore_index=True)
print(f"{len(df_mlp)} runs loaded")
pivot_mlp = to_pivot(df_mlp)
pivot_mlp

83 runs loaded


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
CARL,79.08,86.85,45.21,36.87,37.17,47.33,57.49,30.42,46.00,51.82
Copernicus-FM,80.74,85.06,49.66,29.85,35.75,47.03,57.40,34.68,59.12,53.26
DINOv2,74.10,68.18,57.28,13.94,23.51,19.80,58.27,34.20,62.35,45.74
DOFA,81.40,84.84,52.93,24.45,38.02,50.37,57.03,35.49,66.90,54.60
DOFAv2,74.84,84.24,44.31,19.78,42.30,22.70,57.03,35.15,66.86,49.69
Panopticon,83.38,85.88,66.75,34.48,35.82,43.50,58.50,34.41,66.22,56.55
SMARTIES,81.47,85.26,55.42,33.76,34.93,47.99,49.78,33.82,51.77,52.69
ViT-Frozen,79.83,84.65,32.96,26.58,33.47,49.88,49.78,32.40,41.88,47.94
ViT-Scratch,79.74,84.68,32.76,27.44,35.79,39.73,49.78,32.43,41.93,47.14


## RGB UPerNet results

In [36]:
df_rgb_up = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_rgb" / "upernet",
    metric="test_mIoU",
)

# DINOv2 is RGB-only — pull its results from the full-seg folder
_dinov2_up = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "dinov2_seg",
    metric="test_mIoU",
)
df_rgb_up = pd.concat([df_rgb_up, _dinov2_up], ignore_index=True)

print(f"{len(df_rgb_up)} runs loaded")
pivot_rgb_up = to_pivot(df_rgb_up)
pivot_rgb_up

62 runs loaded


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
CARL,69.78,65.79,37.86,26.11,30.57,36.55,NaN,31.37,42.74,42.60
Copernicus-FM,75.64,74.42,65.11,23.59,37.06,52.60,NaN,35.11,67.70,53.90
DINOv2,74.48,70.02,58.78,18.78,26.58,36.73,62.84,35.20,64.58,49.78
DOFA,76.04,77.26,65.59,24.49,39.82,51.97,NaN,35.51,69.82,55.06
DOFAv2,78.35,75.28,60.86,22.70,34.57,40.09,NaN,38.75,72.47,52.88
Panopticon,75.05,76.62,59.25,28.15,35.59,40.42,NaN,35.38,68.08,52.32
SMARTIES,71.71,68.51,40.07,21.48,28.73,42.05,NaN,32.62,50.38,44.44


## RGB MLP Fusion results

In [37]:
df_rgb_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_rgb" / "mlp_fusion",
    metric="test_mIoU",
)

# DINOv2 is RGB-only — pull its results from the full-seg folder
_dinov2_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "mlp_fusion" / "dinov2_mlp_fusion",
    metric="test_mIoU",
)
df_rgb_mlp = pd.concat([df_rgb_mlp, _dinov2_mlp], ignore_index=True)

print(f"{len(df_rgb_mlp)} runs loaded")
pivot_rgb_mlp = to_pivot(df_rgb_mlp)
pivot_rgb_mlp

57 runs loaded


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
CARL,70.38,63.56,34.48,23.32,27.34,56.58,NaN,27.47,37.30,42.56
Copernicus-FM,75.78,72.62,59.23,22.18,30.81,28.17,NaN,35.21,64.24,48.53
DINOv2,74.10,68.18,57.28,13.94,23.51,19.80,58.27,34.20,62.35,45.74
DOFA,74.79,73.56,57.42,22.28,34.98,28.31,NaN,34.77,66.84,49.12
DOFAv2,76.29,72.51,53.05,21.65,33.17,21.60,NaN,34.95,67.21,47.55
Panopticon,75.49,74.99,61.62,26.60,33.50,46.98,NaN,34.85,64.78,52.35
SMARTIES,71.56,67.62,37.82,18.40,25.68,45.59,NaN,31.02,44.00,42.71


## Advanced usage examples

In [38]:
# Load a single encoder folder
df_panopticon = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "panopticon_seg",
    keep_all_metrics=True,  # get mIoU, mF1, Precision, Recall
)
df_panopticon[["dataset", "test_mIoU", "test_mF1", "test_mPrecision", "test_mRecall"]]

,dataset,test_mIoU,test_mF1,test_mPrecision,test_mRecall
0,DynamicEarthNet,40.553,51.097,51.891,54.459
1,PASTIS,36.814,50.169,46.795,58.957
2,FiveBillionPixels,70.096,80.318,79.304,82.316
3,MADOS,68.217,78.297,78.187,80.500
4,Sen1Floods11,90.591,94.924,94.257,95.619
5,CropTypeMapping,47.994,60.619,65.871,66.839
6,HLS-BurnScars,84.225,90.937,90.860,91.013
7,AI4SmallFarms,37.883,53.821,58.305,56.179
8,SpaceNet7-CD,60.769,68.017,72.581,65.009


In [39]:
# Compare UPerNet vs MLP fusion side-by-side for full-band runs
mean_up  = to_pivot(df_upernet)[["Mean"]].rename(columns={"Mean": "UPerNet"})
mean_mlp = to_pivot(df_mlp)[["Mean"]].rename(columns={"Mean": "MLP Fusion"})
comparison = mean_up.join(mean_mlp, how="outer")
comparison["Δ (UPerNet - MLP)"] = (comparison["UPerNet"] - comparison["MLP Fusion"]).round(2)
comparison.sort_values("UPerNet", ascending=False)

dataset,UPerNet,MLP Fusion,Δ (UPerNet - MLP)
encoder,,,
Panopticon,59.68,56.55,3.13
SMARTIES,57.18,52.69,4.49
UNet,56.35,NaN,NaN
Copernicus-FM,56.27,53.26,3.01
DOFA,55.98,54.60,1.38
DOFAv2,54.09,49.69,4.40
CARL,53.59,51.82,1.77
ViT-Scratch,51.58,47.14,4.44
DINOv2,49.78,45.74,4.04


In [40]:
# Check which runs are still missing (NaN cells)
print("=== Missing UPerNet runs ===")
missing = pivot_upernet.drop(columns="Mean").isna().stack()
print(missing[missing].reset_index()[["encoder", "dataset"]].to_string(index=False))

print("\n=== Missing MLP Fusion runs ===")
missing_mlp = pivot_mlp.drop(columns="Mean").isna().stack()
print(missing_mlp[missing_mlp].reset_index()[["encoder", "dataset"]].to_string(index=False))

=== Missing UPerNet runs ===
Empty DataFrame
Columns: [encoder, dataset]
Index: []

=== Missing MLP Fusion runs ===
Empty DataFrame
Columns: [encoder, dataset]
Index: []


## DOFA / DOFAv2 / ViT-Frozen / DINOv2 comparison

In [41]:
COMPARISON_ENCODERS = ["DOFA", "DOFAv2", "ViT-Frozen", "DINOv2"]

print("── UPerNet ──")
pivot_cmp_up = to_pivot(df_rgb_up, encoder_order=COMPARISON_ENCODERS)
display(pivot_cmp_up)

print("\n── MLP Fusion ──")
pivot_cmp_mlp = to_pivot(df_rgb_mlp, encoder_order=COMPARISON_ENCODERS)
display(pivot_cmp_mlp)

── UPerNet ──


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
DOFA,76.04,77.26,65.59,24.49,39.82,51.97,NaN,35.51,69.82,55.06
DOFAv2,78.35,75.28,60.86,22.70,34.57,40.09,NaN,38.75,72.47,52.88
DINOv2,74.48,70.02,58.78,18.78,26.58,36.73,62.84,35.20,64.58,49.78



── MLP Fusion ──


dataset,HLS-BurnScars,Sen1Floods11,MADOS,PASTIS,DynamicEarthNet,CropTypeMapping,SpaceNet7-CD,AI4SmallFarms,FiveBillionPixels,Mean
encoder,,,,,,,,,,
DOFA,74.79,73.56,57.42,22.28,34.98,28.31,NaN,34.77,66.84,49.12
DOFAv2,76.29,72.51,53.05,21.65,33.17,21.60,NaN,34.95,67.21,47.55
DINOv2,74.10,68.18,57.28,13.94,23.51,19.80,58.27,34.20,62.35,45.74


In [42]:
pivot_cmp_up.to_csv("rgb_dofa_comparison_upernet.csv")

## ViT patch-size ablation: p16 vs p8 (frozen encoder, UPerNet)

In [ ]:
PATCH_SIZE_ENCODERS = ["ViT-Frozen", "ViT-Frozen-p8"]
pivot_patch = to_pivot(df_upernet, encoder_order=PATCH_SIZE_ENCODERS)
pivot_patch

In [43]:
pivot_cmp_mlp.to_csv("rgb_dofa_comparison_mlp_fusion.csv")